In [1]:
import torch
import loralib as lora
import glob
import os
import time
from torchvision.datasets import Food101, ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights
from torchvision.models import EfficientNet_V2_S_Weights
import torchvision.models as models
import pandas as pd
import torch.nn as nn

MODEL_WEIGHTS_PATH = "model_weights"


if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


# =========================================================
# BASIC HELPERS
# =========================================================
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")


def replace_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new_module)


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def to_int(x):
    return x[0] if isinstance(x, tuple) else x


# =========================================================
# DATA
# =========================================================
def get_food101_loaders(transform, batch_size=256, num_workers=2):
    train_dataset = Food101(
        root="data",
        split="train",
        download=True,
        transform=transform
    )

    val_dataset = Food101(
        root="data",
        split="test",
        download=True,
        transform=transform
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_dataloader, val_dataloader

def build_resnet18_lora(num_classes=101, r=8, alpha=16):
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    def to_int(x):
        return x[0] if isinstance(x, tuple) else x

    for name, module in list(model.named_modules()):
        if name in target_layers and isinstance(module, nn.Conv2d):
            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=to_int(module.kernel_size),
                stride=to_int(module.stride),
                padding=to_int(module.padding),
                dilation=to_int(module.dilation),
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )

            new_layer.conv.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.conv.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    lora.mark_only_lora_as_trainable(model)

    for p in model.fc.parameters():
        p.requires_grad = True

    return model

def to_int(x):
    return x[0] if isinstance(x, tuple) else x


def build_efficientnet_v2_s_lora(num_classes=101, r=8, alpha=16):
    model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Conv2d) and name.startswith("features"):
            if module.groups != 1:
                continue

            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=to_int(module.kernel_size),
                stride=to_int(module.stride),
                padding=to_int(module.padding),
                dilation=to_int(module.dilation),
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )

            new_layer.conv.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.conv.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    lora.mark_only_lora_as_trainable(model)

    for p in model.classifier[1].parameters():
        p.requires_grad = True

    return model


def load_model_weights(model_path):
    try:
        state_dict = torch.load(model_path, map_location=device)
        print(f"Successfully loaded {model_path}")
        return state_dict["model_state"]
    except Exception as e:
        print(f"Error loading {model_path}: {e}")
        return None
    


def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    total_time = 0.0  

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.time()
            outputs = model(images)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            total_time += (end - start)

            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    time_per_image = total_time / total
    throughput = total / total_time

    return avg_loss, acc, time_per_image, throughput



class ResNetWithAdapters(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

        self.adapters = nn.ModuleDict({
            "layer1": CNNAdapter(64),
            "layer2": CNNAdapter(128),
            "layer3": CNNAdapter(256),
            "layer4": CNNAdapter(512),
        })

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        x = self.model.layer1(x)
        x = self.adapters["layer1"](x)

        x = self.model.layer2(x)
        x = self.adapters["layer2"](x)

        x = self.model.layer3(x)
        x = self.adapters["layer3"](x)

        x = self.model.layer4(x)
        x = self.adapters["layer4"](x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.fc(x)

        return x
    


class CNNAdapter(nn.Module):
    def __init__(self, channels, reduction_factor=8):
        super().__init__()
        bottleneck = channels // reduction_factor
        self.adapter = nn.Sequential(
            nn.Conv2d(channels, bottleneck, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck, channels, kernel_size=1)
        )

    def forward(self, x):
        return x + self.adapter(x) 
    


class EfficientNetWithAdapters(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

        # pick certain stages to add adapters to. 
        self.adapter_indices = [3, 5, 6]  

        self.adapters = nn.ModuleDict({
            str(i): CNNAdapter(self._get_channels(i))
            for i in self.adapter_indices
        })

    def _get_channels(self, idx):
        # get output channels of that stage
        return self.model.features[idx][-1].out_channels

    def forward(self, x):
        for i, layer in enumerate(self.model.features):
            x = layer(x)
            if str(i) in self.adapters:
                x = self.adapters[str(i)](x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.classifier(x)
        return x

base_model = models.efficientnet_v2_s(
    weights=models.EfficientNet_V2_S_Weights.DEFAULT
)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        # x: (B, C, H, W)

        avg = torch.mean(x, dim=1, keepdim=True)      # (B, 1, H, W)
        max, _ = torch.max(x, dim=1, keepdim=True)    # (B, 1, H, W)

        x_cat = torch.cat([avg, max], dim=1)          # (B, 2, H, W)
        mask = torch.sigmoid(self.conv(x_cat))        # (B, 1, H, W)

        return x * mask

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )

    def forward(self, x):
        b, c, h, w = x.shape

        avg = torch.mean(x, dim=(2, 3))               # (B, C)
        max, _ = torch.max(x.view(b, c, -1), dim=2)   # (B, C)

        attn = self.mlp(avg) + self.mlp(max)
        attn = torch.sigmoid(attn).view(b, c, 1, 1)

        return x * attn

class CBAMAdapter(nn.Module):
    def __init__(self, channels, reduction_factor=8):
        super().__init__()
        
        bottleneck = channels // reduction_factor
        self.conv_adapter = nn.Sequential(
            nn.Conv2d(channels, bottleneck, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck, channels, kernel_size=1)
        )
        # make sure conv adapter starts as identity function
        nn.init.zeros_(self.conv_adapter[-1].weight)
        
        self.spatial_attn = SpatialAttention()
        self.channel_attn = ChannelAttention(channels)

    def forward(self, x):
        x = x + self.conv_adapter(x)
        x = x + self.channel_attn(x)
        x = x + self.spatial_attn(x)
        
        return x

class ResNetWithCBAMAdapters(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

        self.adapters = nn.ModuleDict({
            "layer1": CBAMAdapter(64),
            "layer2": CBAMAdapter(128),
            "layer3": CBAMAdapter(256),
            "layer4": CBAMAdapter(512),
        })

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        x = self.model.layer1(x)
        x = self.adapters["layer1"](x)

        x = self.model.layer2(x)
        x = self.adapters["layer2"](x)

        x = self.model.layer3(x)
        x = self.adapters["layer3"](x)

        x = self.model.layer4(x)
        x = self.adapters["layer4"](x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.fc(x)

        return x

In [2]:

# res_net_model_paths = glob.glob(os.path.join(MODEL_WEIGHTS_PATH, "final_checkpoint_resnet*.pt"))

# res_net_model_paths = sorted(res_net_model_paths)


# transform = ResNet18_Weights.DEFAULT.transforms()
# criterion = torch.nn.CrossEntropyLoss()


# root = "test_splits"

# loaders = {}



# for folder in os.listdir(root):
#     path = os.path.join(root, folder)
#     if not os.path.isdir(path):
#         continue

#     dataset = ImageFolder(root=path, transform=transform)
#     loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=8)

#     loaders[folder] = loader

# res1 = []

# for model_path in res_net_model_paths:
    
#     if "adapter" in model_path:
#         model = ResNetWithAdapters(models.resnet18(weights=None))
#         model.model.fc = torch.nn.Linear(model.model.fc.in_features, 101)
#     elif "lora" in model_path:
#         model = build_resnet18_lora(num_classes=101, r=8, alpha=16)
#     elif "custom" in model_path:
#         model = ResNetWithCBAMAdapters(models.resnet18(weights=None))
#         model.model.fc = torch.nn.Linear(model.model.fc.in_features, 101)
#     else:
#         model = models.resnet18(weights=None)
#         model.fc = torch.nn.Linear(model.fc.in_features, 101)
    
#     model.load_state_dict(load_model_weights(model_path))
#     model.to(device)
#     model.eval()

#     for folder, loader in loaders.items():
        
#         avg_loss, acc, time_per_image, throughput = validate(model, loader, criterion, device)
#         print(
#             f"Model: {model_path}, Folder: {folder}, "
#             f"Val Loss: {avg_loss:.4f}, Val Acc: {acc:.4f}, "
#             f"Time/img: {time_per_image:.6f}, Throughput: {throughput:.2f}"
#             )

#         res1.append((model_path, folder, avg_loss, acc, time_per_image, throughput))

# # saving results to csv
# df1 = pd.DataFrame(res1, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
# df1.to_csv("res_net_results.csv", index=False)

In [3]:
efficient_net_model_paths = glob.glob(os.path.join(MODEL_WEIGHTS_PATH, "final_checkpoint_efficientnet*.pt"))

efficient_net_model_paths = sorted(efficient_net_model_paths)



transform = EfficientNet_V2_S_Weights.DEFAULT.transforms()
criterion = torch.nn.CrossEntropyLoss()

root = "test_splits"

loaders = {}
for folder in os.listdir(root):
    path = os.path.join(root, folder)
    if not os.path.isdir(path):
        continue

    dataset = ImageFolder(root=path, transform=transform)
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=8)
    loaders[folder] = loader

res2 = []

for model_path in efficient_net_model_paths:
    
    if "adapter" in model_path:
        model = EfficientNetWithAdapters(models.efficientnet_v2_s(weights=None))
        in_features = model.model.classifier[1].in_features
        model.model.classifier[1] = nn.Linear(in_features, 101)
        
    elif "lora" in model_path:
        model = build_efficientnet_v2_s_lora(num_classes=101, r=8, alpha=16)
    else:
        model = models.efficientnet_v2_s(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier[1] = torch.nn.Linear(in_features, 101)
    

    model.load_state_dict(load_model_weights(model_path))
    model.to(device)
    model.eval()
    
    for folder, loader in loaders.items():
    
        avg_loss, acc, time_per_image, throughput = validate(model, loader, criterion, device)
        print(
            f"Model: {model_path}, Folder: {folder}, "
            f"Val Loss: {avg_loss:.4f}, Val Acc: {acc:.4f}, "
            f"Time/img: {time_per_image:.6f}, Throughput: {throughput:.2f}"
            )

        res2.append((model_path, folder, avg_loss, acc, time_per_image, throughput))

# saving results to csv
df2 = pd.DataFrame(res2, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
df2.to_csv("efficient_net_results.csv", index=False)

Successfully loaded model_weights/final_checkpoint_efficientnet_batch_norm.pt


Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=94, pipe_handle=122)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniforge/base/envs/cv_env/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/cv_env/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/cv_env/lib/python3.14/site-packages/torchvision/__init__.py", line 8, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniforge/base/envs/cv_env/lib/python3.14/s

KeyboardInterrupt: 

In [ ]:
res3 = res1 + res2
df3 = pd.DataFrame(res3, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
df3.to_csv("combined_results.csv", index=False)


# Grad cam

In [1]:
!pip install -q grad-cam loralib

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import loralib as lora

from torchvision.models import ResNet18_Weights
from torchvision.datasets import Food101

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\Inara\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\Inara\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 87, in _run_code
    ex

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# TODO:
# Replace this with your LOCAL test dataset folder path.
# Example:
# DATA_ROOT = r"C:\Users\Inara\Downloads\block 6\testdataset"
DATA_ROOT = r"ADD_TEST_DATASET_PATH_HERE"

# TODO:
# Replace this with your LOCAL model weights folder path.
# Example:
# MODEL_WEIGHTS_PATH = r"C:\Users\Inara\Downloads\block 6\model_weights"
MODEL_WEIGHTS_PATH = r"ADD_MODEL_WEIGHTS_FOLDER_PATH_HERE"

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
print("MODEL_WEIGHTS_PATH exists:", os.path.exists(MODEL_WEIGHTS_PATH))
print("Checkpoint files found:", glob.glob(os.path.join(MODEL_WEIGHTS_PATH, "*.pt")))

In [ ]:
#food101 class labels

food101_classes = Food101(root="data", split="train", download=True).classes
idx_to_class = {i: c for i, c in enumerate(food101_classes)}
class_to_idx = {c: i for i, c in enumerate(food101_classes)}

print("Number of classes:", len(food101_classes))
print(food101_classes[:10])

In [ ]:
#helpers
def replace_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new_module)

def to_int(x):
    return x[0] if isinstance(x, tuple) else x

In [ ]:
# RESNET18 LORA MODEL

def build_resnet18_lora(num_classes=101, r=8, alpha=16):
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    for name, module in list(model.named_modules()):
        if name in target_layers and isinstance(module, nn.Conv2d):
            new_layer = lora.Conv2d(
                in_channels=module.in_channels,
                out_channels=module.out_channels,
                kernel_size=to_int(module.kernel_size),
                stride=to_int(module.stride),
                padding=to_int(module.padding),
                dilation=to_int(module.dilation),
                groups=module.groups,
                bias=(module.bias is not None),
                r=r,
                lora_alpha=alpha
            )

            new_layer.conv.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_layer.conv.bias.data.copy_(module.bias.data)

            replace_module_by_name(model, name, new_layer)

    lora.mark_only_lora_as_trainable(model)

    for p in model.fc.parameters():
        p.requires_grad = True

    return model

In [ ]:
# RESNET18 TASK-SPECIFIC ADAPTER MODEL
class CNNAdapter(nn.Module):
    def __init__(self, channels, reduction_factor=8):
        super().__init__()
        bottleneck = channels // reduction_factor
        self.adapter = nn.Sequential(
            nn.Conv2d(channels, bottleneck, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck, channels, kernel_size=1)
        )

    def forward(self, x):
        return x + self.adapter(x)

class ResNetWithAdapters(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.adapters = nn.ModuleDict({
            "layer1": CNNAdapter(64),
            "layer2": CNNAdapter(128),
            "layer3": CNNAdapter(256),
            "layer4": CNNAdapter(512),
        })

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        x = self.model.layer1(x)
        x = self.adapters["layer1"](x)

        x = self.model.layer2(x)
        x = self.adapters["layer2"](x)

        x = self.model.layer3(x)
        x = self.adapters["layer3"](x)

        x = self.model.layer4(x)
        x = self.adapters["layer4"](x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.fc(x)
        return x

In [ ]:
# BUILD RESNET18 MODELS FOR GRAD-CAM

def build_resnet18_model(method):
    if method == "linear_probe":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 101)

    elif method == "lora":
        model = build_resnet18_lora(num_classes=101, r=8, alpha=16)

    elif method == "task_specific_adapter":
        model = ResNetWithAdapters(models.resnet18(weights=None))
        model.model.fc = nn.Linear(model.model.fc.in_features, 101)

    else:
        raise ValueError(f"Unsupported method: {method}")

    checkpoint_path = os.path.join(
        MODEL_WEIGHTS_PATH,
        f"final_checkpoint_resnet18_{method}.pt"
    )

    state_dict = load_checkpoint_state(checkpoint_path)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model

In [ ]:
# IMAGE LOADING
def load_image_for_cam(img_path):
    transform = ResNet18_Weights.DEFAULT.transforms()
    pil_img = Image.open(img_path).convert("RGB")
    rgb_img = np.array(pil_img).astype(np.float32) / 255.0
    input_tensor = transform(pil_img).unsqueeze(0)
    return pil_img, rgb_img, input_tensor

In [ ]:
# RUN GRAD-CAM FOR ONE MODEL + ONE IMAGE
def run_gradcam_single(model, method, img_path, use_true_label=False):
    pil_img, rgb_img, input_tensor = load_image_for_cam(img_path)
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax(dim=1).item()

    if use_true_label:
        true_class_name = os.path.basename(os.path.dirname(img_path))
        target_idx = class_to_idx[true_class_name]
    else:
        target_idx = pred_idx

    if method == "task_specific_adapter":
        target_layers = [model.model.layer4[-1]]
    else:
        target_layers = [model.layer4[-1]]

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=[ClassifierOutputTarget(target_idx)]
        )[0]

    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    return pil_img, cam_image, idx_to_class[pred_idx]

In [ ]:
# COMPARE MULTIPLE MODELS ON THE SAME IMAGE
def show_gradcam_comparison(img_path):
    methods = ["linear_probe", "lora", "task_specific_adapter"]
    models_dict = {m: build_resnet18_model(m) for m in methods}

    results = []
    for method in methods:
        pil_img, cam_image, pred_class = run_gradcam_single(
            models_dict[method],
            method,
            img_path
        )
        results.append((method, pil_img, cam_image, pred_class))

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(results[0][1])
    plt.title("Original")
    plt.axis("off")

    for i, (method, _, cam_image, pred_class) in enumerate(results, start=2):
        plt.subplot(1, 4, i)
        plt.imshow(cam_image)
        plt.title(f"{method}\nPred: {pred_class}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# TEST ON ONE CLEAN IMAGE
# This will automatically pick one image from the clean test folder.
clean_images = glob.glob(os.path.join(DATA_ROOT, "clean", "*", "*.jpg"))

print("Found clean images:", len(clean_images))
print("Example clean image:", clean_images[0] if len(clean_images) > 0 else "No image found")

img_path = clean_images[0]
show_gradcam_comparison(img_path)

In [ ]:
# TEST ON ONE BLUR_LITTLE IMAGE
blur_images = glob.glob(os.path.join(DATA_ROOT, "blur_little", "*", "*.jpg"))

print("Found blur_little images:", len(blur_images))
print("Example blur_little image:", blur_images[0] if len(blur_images) > 0 else "No image found")

img_path = blur_images[0]
show_gradcam_comparison(img_path)

In [ ]:
# TEST ON ONE BLUR_MEDIUM IMAGE
blur_medium_images = glob.glob(os.path.join(DATA_ROOT, "blur_medium", "*", "*.jpg"))

print("Found blur_medium images:", len(blur_medium_images))
print("Example blur_medium image:", blur_medium_images[0] if len(blur_medium_images) > 0 else "No image found")

img_path = blur_medium_images[0]
show_gradcam_comparison(img_path)

In [ ]:
# TEST ON ONE NOISE_ROTATION IMAGE
noise_images = glob.glob(os.path.join(DATA_ROOT, "noise_rotation", "*", "*.jpg"))

print("Found noise_rotation images:", len(noise_images))
print("Example noise_rotation image:", noise_images[0] if len(noise_images) > 0 else "No image found")

img_path = noise_images[0]
show_gradcam_comparison(img_path)